# Train MiniReasoner

Prepare the packed corpus with [prepare_data.ipynb](prepare_data.ipynb), then run the cells below in order. The training loop learns next-token prediction and saves a resumable checkpoint. Validation uses held-out stories from the published training split.

## Run settings

Training requires a CUDA GPU. Use the base preset for pretraining or set **preset = "small"** for a shorter GPU run. Automatic precision uses BF16 on supported GPUs and FP32 on GPUs such as the T4. Set **RUN_DIR** in **src/settings.py** for a separate run. To pause at an absolute step, set **stop_after** to that step. To resume, set **resume = True**, keep the same corpus, preset, and checkpoint directory, and rerun from this section.

In [ ]:
import json
import math
import sys
from dataclasses import asdict
from pathlib import Path
from time import perf_counter

import torch

working_dir = Path.cwd()
project_root = next(
    (path for path in (working_dir, *working_dir.parents) if (path / "src" / "settings.py").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Open this notebook from within the MiniReasoner project.")

sys.path.insert(0, str(project_root))
from src.model import GPT, GPTConfig
from src.generate import complete
from src.settings import CHECKPOINT_PATH as checkpoint_path
from src.settings import PRESETS
from src.settings import RUN_DIR as run_dir
from src.settings import TOKENIZED_DATA_DIR as data_dir
from src.tokenizer import TOKEN_DTYPE, sha256
from src.train import (
    TokenStream,
    TrainConfig,
    evaluate,
    load_checkpoint,
    loss_for,
    require_training_gpu,
    resolve_precision,
    save_checkpoint,
)

preset = "base"
train_model = True
inference_device = "cpu"
precision_request = "auto"
stop_after = None
resume = False

if train_model:
    device = require_training_gpu()
    precision = resolve_precision(precision_request, device)

## Load the packed corpus

Check the manifest and file hashes before opening the training and validation streams.

In [ ]:
if train_model:
    settings = PRESETS[preset]
    manifest = json.loads((data_dir / "manifest.json").read_text(encoding="utf-8"))
    if manifest["version"] != 1 or manifest["dtype"] != TOKEN_DTYPE.str:
        raise ValueError("Unsupported dataset format")
    if manifest.get("dataset") == "SimpleStories/SimpleStories" and manifest.get("source_splits") != {
        "train": "train",
        "val": "train",
    }:
        raise ValueError("Prepare SimpleStories with the current notebook so both splits come from the published training data")

    tokenizer_path = data_dir / "tokenizer.json"
    if sha256(tokenizer_path) != manifest["tokenizer_sha256"]:
        raise ValueError("Tokenizer does not match the dataset manifest")

    tokenizer_json = tokenizer_path.read_text(encoding="utf-8")
    vocab_size = manifest["vocab_size"]
    model_config = GPTConfig(vocab_size=vocab_size, **settings["model"])
    print(f"Model: n_embd={model_config.n_embd:,}, n_head={model_config.n_head}, "
          f"head_dim={model_config.n_embd // model_config.n_head}")
    config = TrainConfig(**settings["training"])
    stop = config.steps if stop_after is None else stop_after
    if not 1 <= stop <= config.steps:
        raise ValueError("stop_after must be between 1 and the configured total steps")

    streams = {}
    for split in ("train", "val"):
        path = data_dir / f"{split}.bin"
        size = manifest[split]["tokens"]
        if path.stat().st_size != size * TOKEN_DTYPE.itemsize or sha256(path) != manifest[split]["sha256"]:
            raise ValueError(f"{path} does not match the dataset manifest")
        streams[split] = TokenStream(path, size, model_config.context_length)

    for split in ("train", "val"):
        print(f"{split}: {manifest[split]['documents']:,} stories, {manifest[split]['tokens']:,} tokens")

## Batch and epoch analysis

A full pass below means taking non-overlapping windows of the configured context length. The final batch of that hypothetical pass may be smaller. Training samples random windows with replacement, so its epoch count measures how many token targets it processes relative to the size of the corpus. Validation checks the same evenly spaced windows each time.

In [ ]:
if train_model:
    context_length = model_config.context_length
    batch_size = config.batch_size
    targets_per_train_step = batch_size * context_length

    print(f"{preset.title()} preset: {batch_size:,} windows per batch, {context_length:,} targets per window")
    for split, label in (("train", "Training"), ("val", "Validation")):
        token_count = manifest[split]["tokens"]
        windows = streams[split].windows
        full_batches, last_batch_size = divmod(windows, batch_size)
        pass_batches = full_batches + int(last_batch_size > 0)
        tail_targets = (token_count - 1) % context_length
        print(f"{label}: {token_count:,} packed tokens and {windows:,} non-overlapping windows")
        batch_breakdown = f"{full_batches:,} full batches"
        if last_batch_size:
            batch_breakdown += f" + {last_batch_size:,} windows in the final batch"
        print(f"  One pass: {pass_batches:,} batches ({batch_breakdown})")
        print(f"  Tail outside the full windows: {tail_targets:,} targets")

    train_targets = manifest["train"]["tokens"] - 1
    scheduled_targets = stop * targets_per_train_step
    print(f"\nTraining from scratch through step {stop:,}: "
          f"{scheduled_targets:,} targets = {scheduled_targets / train_targets:.4f} epoch equivalents")

    val_windows = min(config.eval_windows, streams["val"].windows)
    val_batches = (val_windows + batch_size - 1) // batch_size
    val_fraction = val_windows * context_length / (manifest["val"]["tokens"] - 1)
    fresh_run_checks = 1 + (stop + config.eval_every - 1) // config.eval_every
    batch_label = "batch" if val_batches == 1 else "batches"
    print(f"Validation per check: {val_windows:,} windows in {val_batches:,} {batch_label}, "
          f"covering {val_fraction:.2%} of validation targets ({val_fraction:.4f} of one pass)")
    print(f"Fresh-run schedule through step {stop:,}: {fresh_run_checks:,} checks, including step 0")

## Initialize the model or resume a checkpoint

Set up AdamW and the learning-rate schedule. Resuming restores the optimizer, batch generator, and loss history.

In [ ]:
if train_model:
    if resume and not checkpoint_path.is_file():
        raise FileNotFoundError(f"No checkpoint to resume at {checkpoint_path}.")

    if not resume and checkpoint_path.exists():
        raise FileExistsError(
            f"Checkpoint already exists: {checkpoint_path}. Set resume = True to continue.")

    torch.manual_seed(config.seed)
    batch_rng = torch.Generator(device="cpu").manual_seed(config.seed)
    model = GPT(model_config).to(device)
    decay, no_decay = [], []

    for parameter in model.parameters():
        (decay if parameter.ndim >= 2 else no_decay).append(parameter)

    optimizer = torch.optim.AdamW(
        [{"params": decay}, {"params": no_decay, "weight_decay": 0.0}],
        lr=config.learning_rate,
        weight_decay=config.weight_decay,
        betas=(0.9, 0.95),
        fused=True,
    )
    tokens_per_step = config.batch_size * model_config.context_length
    run_identity = {
        "model_config": asdict(model_config),
        "train_config": asdict(config),
        "data_manifest": manifest,
    }
    start = 0
    if resume:
        checkpoint = load_checkpoint(checkpoint_path)
        if checkpoint["version"] != 1 or any(checkpoint.get(k) != v for k, v in run_identity.items()):
            raise ValueError(
                "Resume requires matching training settings and data. "
                "For a fresh run, select a new checkpoint path and set resume=False."
            )
        start = checkpoint["step"]
        if stop <= start:
            raise ValueError(
                f"stop_after must exceed the completed step ({start})")
        model.load_state_dict(checkpoint["model"])
        optimizer.load_state_dict(checkpoint["optimizer"])
        batch_rng.set_state(checkpoint["batch_rng"])
        history = checkpoint["metrics"]
        del checkpoint
    else:
        history = []

    training_started = False
    num_parameters = sum(parameter.numel() for parameter in model.parameters())
    print(
        f"{num_parameters:,} parameters | {device} {precision} | "
        f"{tokens_per_step:,} tokens/step | steps {start + 1} to {stop}",
        flush=True,
    )

## Train and validate

Each optimizer step runs one batch of next-token predictions. The loop logs training and validation loss and saves a resumable checkpoint at each evaluation interval and at the requested stop step. After an interruption, use the resume settings to continue from the last saved checkpoint.

In [ ]:
if train_model:
    if training_started:
        raise RuntimeError("Rerun the settings and initialization cells before starting training again.")

    if not resume:
        run_dir.mkdir(parents=True, exist_ok=True)
        if checkpoint_path.exists():
            raise FileExistsError(f"Checkpoint already exists: {checkpoint_path}")

    training_started = True
    if not resume:
        initial_loss = evaluate(model, streams["val"], config, device, precision)
        history = [{"step": 0, "tokens": 0, "val_loss": initial_loss}]
        print(f"step 0 | val loss {initial_loss:.4f}", flush=True)

    interval_start = perf_counter()
    interval_steps = 0
    running_loss = torch.zeros((), device=device)
    for step in range(start + 1, stop + 1):
        lr = config.lr_at(step)
        for group in optimizer.param_groups:
            group["lr"] = lr
        optimizer.zero_grad(set_to_none=True)
        starts = torch.randint(
            len(streams["train"].tokens) - model_config.context_length,
            (config.batch_size,),
            generator=batch_rng,
            device="cpu",
        ).numpy()
        x, y = streams["train"].batch(starts, device)
        loss = loss_for(model, x, y, precision)
        running_loss += loss.detach()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        interval_steps += 1

        if step % config.eval_every != 0 and step != stop:
            continue
        train_loss = running_loss.item() / interval_steps
        throughput = interval_steps * tokens_per_step / (perf_counter() - interval_start)
        val_loss = evaluate(model, streams["val"], config, device, precision)
        if not math.isfinite(train_loss) or not math.isfinite(val_loss):
            raise FloatingPointError("Non-finite loss. The last valid checkpoint is preserved.")
        metrics = {
            "step": step,
            "tokens": step * tokens_per_step,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "lr": lr,
            "train_tokens_per_second": round(throughput),
        }
        history.append(metrics)
        save_checkpoint(
            checkpoint_path,
            {
                "version": 1,
                **run_identity,
                "step": step,
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "batch_rng": batch_rng.get_state(),
                "tokenizer_json": tokenizer_json,
                "metrics": history,
            },
        )
        print(json.dumps(metrics), flush=True)
        running_loss.zero_()
        interval_steps = 0
        interval_start = perf_counter()
    print(f"Checkpoint saved to {checkpoint_path}", flush=True)

## Sample from the saved checkpoint

For inference only, set **train_model = False** and run the notebook with a saved checkpoint. Set the checkpoint path through **RUN_DIR** in **src/settings.py**. The default **inference_device = "cpu"** lets you generate text on CPU. Set it to **"cuda"** for GPU inference. Edit the prompt and token limit in the cell below.

In [ ]:
prompt = "Hi there! How are you doing today?"
max_new_tokens = 24

completion = complete(checkpoint_path, prompt, max_new_tokens, torch.device(inference_device))
print(prompt + completion)